# Siemens MRRT Questionnaire - Exploratory Data Analysis

**Purpose:** Analyze Siemens MRRT questionnaire data about MR imaging usage in radiation therapy

**Data Source:** `data/raw/siemens/` (questionnaire responses in wide and long format)

**Key Questions:**
- What percentage of patients are planned with MR images?
- Which anatomical sites use MR most commonly?
- What are the main pain points in MR-guided radiotherapy?
- What is the pricing sensitivity (Van Westendorp model)?
- Which MR equipment vendors are being used?

In [ ]:
# Imports
import sys
sys.path.append('../')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Imports successful!")

## 1. Load Data

In [ ]:
# Load processed data
wide_df = pd.read_csv('../data/processed/siemens/processed_wide_format.csv')
long_df = pd.read_csv('../data/processed/siemens/processed_long_format.csv')

print(f"Wide format: {wide_df.shape}")
print(f"Long format: {long_df.shape}")
print(f"\nUnique responses: {long_df['response_id'].nunique()}")

## 2. Data Overview

In [ ]:
# Display first few rows of wide format
print("Wide Format Sample:")
display(wide_df.head())

print("\nColumn names (first 20):")
print(wide_df.columns.tolist()[:20])

In [ ]:
# Display first few rows of long format
print("Long Format Sample:")
display(long_df.head(20))

## 3. Response Demographics

In [ ]:
# Country distribution
country_counts = long_df.groupby('response_id')['country'].first().value_counts()

plt.figure(figsize=(10, 6))
country_counts.plot(kind='bar', color='steelblue')
plt.title('Responses by Country', fontsize=14, fontweight='bold')
plt.xlabel('Country')
plt.ylabel('Number of Responses')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\nCountry Distribution:")
print(country_counts)

In [ ]:
# Institution names
institutions = long_df.groupby('response_id')['institution_name'].first()
print("Institutions surveyed:")
for resp_id, inst in institutions.items():
    print(f"  {resp_id}: {inst}")

## 4. Section Analysis

Analyze responses by questionnaire section

In [ ]:
# Responses by section
section_counts = long_df['section'].value_counts()

plt.figure(figsize=(12, 6))
section_counts.plot(kind='bar', color='coral')
plt.title('Number of Responses by Questionnaire Section', fontsize=14, fontweight='bold')
plt.xlabel('Section')
plt.ylabel('Number of Responses')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\nSection breakdown:")
print(section_counts)

## 5. Equipment & MR Vendor Analysis

In [ ]:
# MR vendor analysis
mr_vendors = long_df[long_df['question_text'].str.contains('MR Vendor', na=False)]

if not mr_vendors.empty:
    print("MR Equipment Vendors:")
    vendor_counts = mr_vendors['response_text'].value_counts()
    print(vendor_counts)
    
    plt.figure(figsize=(10, 6))
    vendor_counts.plot(kind='barh', color='teal')
    plt.title('MR Equipment Vendors', fontsize=14, fontweight='bold')
    plt.xlabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print("No MR vendor data found in current format")

In [ ]:
# Equipment counts (LINACs, CT simulators, etc.)
equipment_data = long_df[long_df['section'] == 'background'].copy()

print("Equipment-related questions:")
print(equipment_data['question_text'].unique()[:10])

## 6. Clinical Sites Analysis

In [ ]:
# Clinical sites section
clinical_sites = long_df[long_df['section'] == 'clinical_sites'].copy()

print(f"Clinical site responses: {len(clinical_sites)}")
print("\nSample clinical site questions:")
display(clinical_sites.head(15))

In [ ]:
# Analyze which clinical sites are most commonly treating with MR
treating_sites = clinical_sites[clinical_sites['subsection'] == 'treating']

if not treating_sites.empty:
    # Count true values per site
    site_summary = treating_sites.groupby('question_text')['response_value'].apply(
        lambda x: (x == 'true').sum()
    ).sort_values(ascending=False)
    
    print("Clinical sites being treated (sorted by frequency):")
    print(site_summary)
    
    plt.figure(figsize=(12, 6))
    site_summary.plot(kind='barh', color='mediumseagreen')
    plt.title('Clinical Sites Treating with MR', fontsize=14, fontweight='bold')
    plt.xlabel('Number of Institutions')
    plt.tight_layout()
    plt.show()

## 7. Pain Points Analysis

In [ ]:
# Search for pain-related questions
pain_keywords = ['pain', 'challenge', 'issue', 'problem', 'difficulty']
pain_related = long_df[
    long_df['question_text'].str.lower().str.contains('|'.join(pain_keywords), na=False)
]

if not pain_related.empty:
    print(f"Found {len(pain_related)} pain-related responses")
    print("\nPain-related questions:")
    print(pain_related['question_text'].unique())
    
    print("\nPain point responses:")
    display(pain_related[['response_id', 'country', 'question_text', 'response_text']].head(20))
else:
    print("No pain points section found in this dataset.")

## 8. Pricing Analysis (Van Westendorp Model)

In [ ]:
# Extract pricing data from wide format
pricing_cols = [col for col in wide_df.columns if 'pricing' in col.lower()]

if pricing_cols:
    print(f"Found pricing columns: {pricing_cols}")
    print("\nPricing data:")
    display(wide_df[['response_id', 'country'] + pricing_cols])
    
    # Convert to numeric and analyze
    pricing_df = wide_df[pricing_cols].apply(pd.to_numeric, errors='coerce')
    
    print("\nPricing statistics:")
    print(pricing_df.describe())
    
    # Plot Van Westendorp curve
    if not pricing_df.empty:
        fig, ax = plt.subplots(figsize=(12, 6))
        for col in pricing_cols:
            if col in pricing_df.columns:
                pricing_df[col].plot(kind='kde', ax=ax, label=col.replace('pricing_', ''))
        
        ax.set_title('Van Westendorp Price Sensitivity', fontsize=14, fontweight='bold')
        ax.set_xlabel('Price (EUR)')
        ax.set_ylabel('Density')
        ax.legend()
        plt.tight_layout()
        plt.show()
else:
    print("No pricing data found in wide format.")

## 9. Key Metrics Summary

In [ ]:
# Summary statistics
print("="*60)
print("KEY FINDINGS SUMMARY")
print("="*60)

print(f"\nTotal unique responses: {long_df['response_id'].nunique()}")
print(f"Countries represented: {long_df['country'].nunique()}")
print(f"Institutions surveyed: {long_df['institution_name'].nunique()}")
print(f"Total data points (long format): {len(long_df)}")
print(f"Questionnaire sections: {long_df['section'].nunique()}")

print("\nData completeness by column:")
completeness = (1 - long_df.isnull().sum() / len(long_df)) * 100
print(completeness[completeness < 100].round(2))

## 10. Export Key Insights

In [ ]:
# Create insights directory
insights_dir = Path('../data/processed/siemens/insights')
insights_dir.mkdir(exist_ok=True)

# Save section summaries
section_summary = long_df.groupby('section').agg({
    'response_id': 'count',
    'question_id': 'nunique'
}).rename(columns={'response_id': 'total_responses', 'question_id': 'unique_questions'})

section_summary.to_csv(insights_dir / 'section_summary.csv')
print(f"Saved section summary to {insights_dir / 'section_summary.csv'}")

# Save pain points if found
if not pain_related.empty:
    pain_related.to_csv(insights_dir / 'pain_points.csv', index=False)
    print(f"Saved pain points to {insights_dir / 'pain_points.csv'}")

print("\nEDA Complete!")